In [ ]:
from dq_network import QNetwork, ReplayBuffer 
from mdp import WarehouseMDP
import math 

: 

In [ ]:
# Setup for Google Colab 
# Imports
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from collections import deque
from torch.optim.lr_scheduler import MultiStepLR
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/rl-project-warhouse-nav/')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
!cp '/content/drive/MyDrive/rl-project-warhouse-nav/dq_network.py'      /content/dq_network.py
!cp '/content/drive/MyDrive/rl-project-warhouse-nav/mdp.py'      /content/mdp.py

# Imports 
from mdp import WarehouseMDP
from dq_network import QNetwork, ReplayBuffer

In [ ]:
mdp = WarehouseMDP(
    grid=[
        [0, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
    ],
    start_pos=(0, 0),
    packages={0: (0, 1), 1: (3, 3)},
    storages={0: (3, 0), 1: (3, 2)},
)

In [ ]:
def encode_state(state, n_packages):
    """
    encoding given state s = (robot_pos, carrying, delivered)
    to vector so it can be feeded to Network
    """
    pos, carrying, delivered = state
    row, col = pos
    carry_val = -1.0 if carrying is None else float(carrying)
    # bool b for d in delivered => 1 if true 0 else 
    return np.array([
        float(row), float(col), carry_val] + [float(d) for d in delivered], 
        dtype=np.float32
    )

In [ ]:
# define online network and target network 
# input dim is defined as every possible configuration the robot can be in
# state s = (robot_pos, carrying, delivered)
# encoded by vector: 
#   -  robot_pos (row, col) 
#   -  carrying None = not carrying; int = package id (only one package allowed to carry at time t)
#   -  delivered = one bool per package  
# => input_dim = 3 + num_packages 
input_dim = 3 + mdp.n_packages
policy_network = QNetwork(input_dim = input_dim, output_dim = mdp.n_actions, hidden_dim = 256).to(device)
target_network = QNetwork(input_dim = input_dim, output_dim = mdp.n_actions, hidden_dim = 256).to(device)
policy_optimizer = torch.optim.Adam(params=policy_network.parameters(), lr=5e-4)

In [ ]:
def parameter_update(source_model, target_model, tau):
    for target_param, source_param in zip(target_model.parameters(), source_model.parameters()):
        target_param.data.copy_(tau * source_param.data + (1.0 - tau)*target_param.data)

In [ ]:
# setup according to PyTorch tutorial 
#  
BATCH_SIZE = 128
GAMMA = 0.99
# probability of choosing random action starts with EPS_START
#  and decays exponentially towards EPS_END.
EPS_START = 0.9 
EPS_END = 0.01
# Controls the rate of decay 
EPS_DECAY = 2500
TAU = 0.005
LR = 3e-4
SOFT_UPDATE = 0.01


NUM_TRAJECTORIES = 2000
MAX_EPISODE_LENGTH = 500

# warmup steps to collect the data first
WARMUP = 1000

# placeholders for rewards for each episode
cumulative_rewards = []
policy_losses = []
# declaring the replay buffer
transition_buffer = ReplayBuffer(10000)
steps_done = 0

## Training Loop 

In [ ]:

# iterating through trajectories
for tau in tqdm(range(NUM_TRAJECTORIES)):
    # resetting the environment
    # i.e robot_pos = start_pos, carrying = None, delivered = false for each package in n_packages
    state = mdp.reset()
    # encode state
    state_enc = encode_state(state, mdp.n_packages)
    # setting done to False for while loop 
    done = False
    t = 0
    episode_reward = 0.0 
    while done == False and t < MAX_EPISODE_LENGTH:
        eps_threshold = EPS_END + (EPS_START - EPS_END) * math.exp(-steps_done/EPS_DECAY)
        steps_done += 1 
        # retrieving Q(s, a) for all possible a
        action_q_values = policy_network(torch.tensor(state_enc).to(device))
        # epsilon-greedy action
        action = np.random.choice(
            [torch.argmax(action_q_values.flatten()).detach().cpu().numpy(), 
             np.random.randint(mdp.n_actions)], 
             p=[1-eps_threshold, eps_threshold])
        # keeping track of previous state
        prev_state_enc = state_enc
        # environment step
        next_state, reward, done = mdp.step(state, int(action))
        next_state_enc = encode_state(next_state, mdp.n_packages)
        episode_reward += reward


        transition_buffer.append((prev_state_enc, action, reward, next_state_enc, done))
        

        t += 1
        state = next_state
        state_enc = next_state_enc

    cumulative_rewards.append(episode_reward) 

    if len(transition_buffer) > WARMUP:
        # buffer returns states in encoded format 
        states, actions, rewards, next_states, dones = transition_buffer.sample(sample_size=BATCH_SIZE)
        # max(x) return first x values ordered in a decreasing order
        q_target = target_network(torch.tensor(next_states).to(device)).detach().max(1)[0]
        # using Q-values of target network only for non-terminal state
        expected_values = rewards + GAMMA * q_target*(torch.ones(BATCH_SIZE).to(device) - dones)
        # selecting Q-values of actions taken, using current policy network
        # gather() takes only values indicated by a given index, in this case, action taken
        output = policy_network(states).gather(1, actions.view(-1,1))
        # computing the loss between r + γ * max Q(s',a) and Q(s,a)
        loss = F.mse_loss(output.flatten(), expected_values)
        policy_losses.append(loss.item())
        policy_optimizer.zero_grad()
        loss.backward()
        policy_optimizer.step()
        # decrease the learning rate by factor gamma = 0.1
        policy_scheduler.step()
        # soft parameter update
        parameter_update(policy_network, target_network, SOFT_UPDATE)


In [ ]:
# plot results 

# running mean function for the purpose of visualization
def running_mean(x, N):
    cumsum = np.cumsum(np.insert(x, 0, 0)) 
    return (cumsum[N:] - cumsum[:-N]) / float(N)

plt.figure(figsize=(12, 9))
plt.plot(running_mean(cumulative_rewards, 50), label="DQN (with LR scheduling)")
plt.title("DQN Training — Warehouse Navigation (running mean over 50 episodes)")
plt.xlabel("Episode")
plt.ylabel("Cumulative Reward")
plt.grid()
plt.legend()
plt.savefig("warehouse_rewards.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd

df = pd.DataFrame(cumulative_rewards, columns=['reward'])
df.to_csv('DQN.csv')